In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Feature store exists?:", os.path.exists(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet"))

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda
Feature store exists?: True


In [2]:
import numpy as np
import pandas as pd

df = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet")

df["departure_datetime_local"] = pd.to_datetime(df["departure_datetime_local"])
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["route_id","departure_datetime_local"]).reset_index(drop=True)
df.head()

,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,pax_lag_1w,pax_lag_2w,veh_lag_1w,veh_lag_2w,pax_roll_mean_20,pax_roll_std_20,veh_roll_mean_20,veh_roll_std_20,delay_roll_mean_20,occ_pax_roll_mean_20
0,LevanteFerries,T0000075,DEN-FOR,Denia,Formentera,2024-01-07 08:00:00,2024-01-07,08:00,6,1,...,542.0,533.0,194.0,160.0,581.25,157.994295,173.80,44.684390,5.60,0.645833
1,LevanteFerries,T0000076,DEN-FOR,Denia,Formentera,2024-01-07 12:00:00,2024-01-07,12:00,6,1,...,631.0,528.0,214.0,162.0,587.10,156.085638,175.95,43.950301,5.60,0.652333
2,LevanteFerries,T0000077,DEN-FOR,Denia,Formentera,2024-01-07 17:00:00,2024-01-07,17:00,6,1,...,591.0,399.0,168.0,114.0,585.10,155.248494,174.35,43.221918,6.40,0.650111
3,LevanteFerries,T0000078,DEN-FOR,Denia,Formentera,2024-01-07 20:00:00,2024-01-07,20:00,6,1,...,733.0,393.0,192.0,142.0,588.70,155.762674,173.30,43.510555,5.75,0.654111
4,LevanteFerries,T0000089,DEN-FOR,Denia,Formentera,2024-01-08 08:00:00,2024-01-08,08:00,0,1,...,872.0,474.0,301.0,122.0,601.20,156.441211,177.30,42.798303,5.75,0.501000


In [3]:
def build_baseline(d):
    out = d.copy()

    # Slot = ruta + weekday + hora de salida
    out["dep_key"] = out["route_id"].astype(str) + "|" + out["weekday"].astype(str) + "|" + out["dep_time"].astype(str)

    # Lookup para el año anterior (aprox 364 días)
    lookup = out[["dep_key","date","pax_real"]].rename(columns={"pax_real":"pax_ly_slot"})
    out["date_ly"] = out["date"] - pd.Timedelta(days=364)

    out = out.merge(
        lookup,
        left_on=["dep_key","date_ly"],
        right_on=["dep_key","date"],
        how="left",
        suffixes=("","_drop")
    )
    out = out.drop(columns=["date_drop"], errors="ignore")

    # Ajuste de tendencia:
    # Ratio rolling actual vs rolling LY (clamp para estabilidad)
    lookup_roll = out[["dep_key","date","pax_roll_mean_20"]].rename(columns={"pax_roll_mean_20":"roll_20_ly"})
    out = out.merge(
        lookup_roll,
        left_on=["dep_key","date_ly"],
        right_on=["dep_key","date"],
        how="left",
        suffixes=("","_drop2")
    )
    out = out.drop(columns=["date_drop2"], errors="ignore")

    out["trend_factor"] = (out["pax_roll_mean_20"] / out["roll_20_ly"]).replace([np.inf, -np.inf], np.nan)
    out["trend_factor"] = out["trend_factor"].fillna(1.0).clip(0.75, 1.35)

    base = out["pax_ly_slot"].fillna(out["pax_lag_1w"])
    out["pax_pred_baseline"] = (base * out["trend_factor"]).clip(lower=0)

    # Cap por capacidad (en ferry esto es obligatorio)
    out["pax_pred_baseline"] = np.minimum(out["pax_pred_baseline"], out["capacity_pax"])

    return out

baseline_df = build_baseline(df)
baseline_df[["route_id","departure_datetime_local","pax_real","pax_pred_baseline","capacity_pax"]].head(10)

,route_id,departure_datetime_local,pax_real,pax_pred_baseline,capacity_pax
0,DEN-FOR,2024-01-07 08:00:00,594,542.0,900
1,DEN-FOR,2024-01-07 12:00:00,628,631.0,900
2,DEN-FOR,2024-01-07 17:00:00,644,591.0,900
3,DEN-FOR,2024-01-07 20:00:00,728,733.0,900
4,DEN-FOR,2024-01-08 08:00:00,435,872.0,1200
5,DEN-FOR,2024-01-08 12:00:00,556,900.0,900
6,DEN-FOR,2024-01-08 20:00:00,503,900.0,1200
7,DEN-FOR,2024-01-09 08:00:00,325,594.0,900
8,DEN-FOR,2024-01-09 17:00:00,429,628.0,900
9,DEN-FOR,2024-01-09 20:00:00,475,644.0,900


In [4]:
def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

cutoff = pd.Timestamp("2025-07-01")
test = baseline_df[baseline_df["departure_datetime_local"] >= cutoff].copy()

y = test["pax_real"].values
p = test["pax_pred_baseline"].values

metrics = {
    "cutoff": str(cutoff.date()),
    "MAE": float(np.mean(np.abs(y - p))),
    "WAPE": float(wape(y, p)),
    "n_test": int(len(test))
}
metrics

{'cutoff': '2025-07-01',
 'MAE': 90.33067187281323,
 'WAPE': 0.1293094122891677,
 'n_test': 2346}

In [5]:
per_route = (
    test.groupby("route_id")
    .apply(lambda g: pd.Series({
        "MAE": np.mean(np.abs(g["pax_real"] - g["pax_pred_baseline"])),
        "WAPE": wape(g["pax_real"].values, g["pax_pred_baseline"].values),
        "n": len(g)
    }))
    .reset_index()
    .sort_values("WAPE")
)

display(per_route)

baseline_df.to_parquet(f"{PROJECT_ROOT}/data/processed/baseline_predictions.parquet", index=False)
per_route.to_csv(f"{PROJECT_ROOT}/reports/baseline_per_route_metrics.csv", index=False)

print("Saved ✅ baseline_predictions.parquet + baseline_per_route_metrics.csv")

/tmp/ipykernel_426/2096868853.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,route_id,MAE,WAPE,n
1,DEN-IBZ,87.582160,0.110079,665.0
0,DEN-FOR,84.151381,0.117720,665.0
3,VAL-IBZ,83.363893,0.149772,508.0
2,DEN-PMI,108.984436,0.157460,508.0


Saved ✅ baseline_predictions.parquet + baseline_per_route_metrics.csv
